# 3  Estimation of embeddings

This notebook intended to prepare embeddings for futher reindexing and evaluation.

In [1]:
import os
import pandas as pd
from glob import glob
from tqdm.notebook import tqdm
from PIL import Image

from common.configuration import ModelPath, DatasetPath, species_invasive, species_native
from pipeline.full_predict import ImageProcessor
from common.configuration import DatasetPath, ModelPath, species_native, species_label_map_all, species_classes_map_all

In [2]:
model_class_to_c_number_remap = {}
for i, c in enumerate(species_native):
    model_class_to_c_number_remap[i] = species_classes_map_all[c]

def prediction_to_class_name(prediction):
    predicted_c_number = model_class_to_c_number_remap.get(prediction, -1)
    return species_label_map_all[predicted_c_number]

In [3]:
image_processor = ImageProcessor(ModelPath.instance_segmentation, ModelPath.osc_classification, prediction_to_class_name)

In [4]:
def process_set(images_ds: pd.DataFrame):
    predictions = []
    for ind, item in tqdm(images_ds.iterrows()):
        pred = item.to_dict()
        with Image.open(item.full_path) as obj:
            pred_class, embeddings = image_processor._classify_image(
                    obj, augment=0, threshold=8.0)
            pred["pred_class"] = pred_class
            pred["embeddings"] = embeddings
            predictions.append(pred)

    predictions_df = pd.DataFrame(predictions)
    return predictions_df

def unpack_embeddings_and_save(df, path, col_to_unpack = "embeddings"):
    unpacked = pd.DataFrame(df[col_to_unpack].tolist(), columns=range(len(df[col_to_unpack].iloc[0])))
    result = pd.concat([df.drop(columns=[col_to_unpack]), unpacked], axis=1)
    result.to_csv(path)

## Constansts and paths

In [5]:
trainval_cls_pred_path = "Save/trainval_classification_predictions.csv"
test_cls_pred_path = "Save/test_classification_predictions.csv"
hist_cls_pred_path = "Save/historical_classification_predictions.csv"

test_subset_csv_path = "Save/classification_test_set.csv"

## Reference test subset

In [6]:
test_subset_pd = pd.read_csv(test_subset_csv_path, index_col=0)
test_subset_pd

,label,image
0,Acer,Dataset/pollen_dataset_kfu_2025\reference_ds\A...
1,Acer,Dataset/pollen_dataset_kfu_2025\reference_ds\A...
2,Acer,Dataset/pollen_dataset_kfu_2025\reference_ds\A...
3,Acer,Dataset/pollen_dataset_kfu_2025\reference_ds\A...
4,Acer,Dataset/pollen_dataset_kfu_2025\reference_ds\A...
...,...,...
1463,Malus,Dataset/pollen_dataset_kfu_2025\reference_ds\M...
1464,Malus,Dataset/pollen_dataset_kfu_2025\reference_ds\M...
1465,Malus,Dataset/pollen_dataset_kfu_2025\reference_ds\M...
1466,Malus,Dataset/pollen_dataset_kfu_2025\reference_ds\M...


In [7]:
test_subset_paths = [os.path.normpath(p) for p in test_subset_pd.image.to_list()]
test_subset_paths = set(test_subset_paths)

In [8]:
test_images_ds = []
for img in test_subset_paths:
    test_images_ds.append(
        {
            "full_path" : img,
            "file_name" : os.path.split(img)[1],
            "microscopy_name" : os.path.split(img)[1].rsplit("_", 1)[0],
            "label"  : os.path.split(os.path.split(img)[0])[1]
        }
    )
test_images_ds = pd.DataFrame(test_images_ds)
test_images_ds

,full_path,file_name,microscopy_name,label
0,Dataset\pollen_dataset_kfu_2025\reference_ds\B...,betula (23)_5.png,betula (23),Betula
1,Dataset\pollen_dataset_kfu_2025\reference_ds\M...,Malus (82)_0.png,Malus (82),Malus
2,Dataset\pollen_dataset_kfu_2025\reference_ds\M...,Malus (33)_1.png,Malus (33),Malus
3,Dataset\pollen_dataset_kfu_2025\reference_ds\M...,Malus2 (6)_1.png,Malus2 (6),Malus
4,Dataset\pollen_dataset_kfu_2025\reference_ds\A...,apiaceae (131)_0.png,apiaceae (131),Apiaceae
...,...,...,...,...
1463,Dataset\pollen_dataset_kfu_2025\reference_ds\A...,apiaceae (92)_1.png,apiaceae (92),Apiaceae
1464,Dataset\pollen_dataset_kfu_2025\reference_ds\C...,chenopodium (93)_11.png,chenopodium (93),Chenopodiaceae
1465,Dataset\pollen_dataset_kfu_2025\reference_ds\P...,31-05-25 (83)_0.png,31-05-25 (83),Pinus
1466,Dataset\pollen_dataset_kfu_2025\reference_ds\M...,Malus (56)_0.png,Malus (56),Malus


In [9]:
test_predictions_df = process_set(test_images_ds)
test_predictions_df.to_csv(test_cls_pred_path)
unpack_embeddings_and_save(test_predictions_df, test_cls_pred_path)

0it [00:00, ?it/s]

i:\git\Pollen-Analyzer\common\osc_classification.py:94: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4419.)
  scores[range(scores.shape[0]), candidate.T] < threshold, candidate, -1)


## Train and validation subset

In [10]:
all_reference_images = glob(os.path.join(DatasetPath.reference_images, "*", "*.png"))
all_reference_images = [os.path.normpath(p) for p in all_reference_images]
all_reference_images = set(all_reference_images)
trainval_reference_images = all_reference_images - test_subset_paths

In [11]:
trainval_images_ds = []
for img in trainval_reference_images:
    trainval_images_ds.append(
        {
            "full_path" : img,
            "file_name" : os.path.split(img)[1],
            "microscopy_name" : os.path.split(img)[1].rsplit("_", 1)[0],
            "label"  : os.path.split(os.path.split(img)[0])[1]
        }
    )
trainval_images_ds = pd.DataFrame(trainval_images_ds)
trainval_images_ds

,full_path,file_name,microscopy_name,label
0,Dataset\pollen_dataset_kfu_2025\reference_ds\A...,artemisia (106)_0.png,artemisia (106),Artemisia
1,Dataset\pollen_dataset_kfu_2025\reference_ds\S...,23-04-25 (87)_4.png,23-04-25 (87),Salix
2,Dataset\pollen_dataset_kfu_2025\reference_ds\C...,chenopodium (92)_16.png,chenopodium (92),Chenopodiaceae
3,Dataset\pollen_dataset_kfu_2025\reference_ds\T...,Tilia_40X (79)_1.png,Tilia_40X (79),Tilia
4,Dataset\pollen_dataset_kfu_2025\reference_ds\C...,chenopodium (106)_2.png,chenopodium (106),Chenopodiaceae
...,...,...,...,...
7651,Dataset\pollen_dataset_kfu_2025\reference_ds\Q...,Quercus_40X (12)_2.png,Quercus_40X (12),Quercus
7652,Dataset\pollen_dataset_kfu_2025\reference_ds\A...,Acer_40X (64)_0.png,Acer_40X (64),Acer
7653,Dataset\pollen_dataset_kfu_2025\reference_ds\P...,salix (1)_1.png,salix (1),Plantago
7654,Dataset\pollen_dataset_kfu_2025\reference_ds\P...,Poaceae (41)_0.png,Poaceae (41),Poaceae


In [12]:
trainval_predictions_df = process_set(trainval_images_ds)
unpack_embeddings_and_save(trainval_predictions_df, trainval_cls_pred_path)

0it [00:00, ?it/s]

## Historical test subset

In [13]:
all_hist_images = glob(os.path.join(DatasetPath.real_historical_samples_classification, "*", "*.png"))

In [14]:
hist_images_ds = []
for img in all_hist_images:
    hist_images_ds.append(
        {
            "full_path" : img,
            "file_name" : os.path.split(img)[1],
            "microscopy_name" : os.path.split(img)[1].rsplit("_", 1)[0],
            "label"  : os.path.split(os.path.split(img)[0])[1]
        }
    )
hist_images_ds = pd.DataFrame(hist_images_ds)
hist_images_ds

,full_path,file_name,microscopy_name,label
0,Dataset/pollen_dataset_kfu_2025\historical_cla...,01 jul (152)_0.png,01 jul (152),0_Not pollen
1,Dataset/pollen_dataset_kfu_2025\historical_cla...,01 jul (210)_0.png,01 jul (210),0_Not pollen
2,Dataset/pollen_dataset_kfu_2025\historical_cla...,01 jul (230)_0.png,01 jul (230),0_Not pollen
3,Dataset/pollen_dataset_kfu_2025\historical_cla...,01 jul (231)_0.png,01 jul (231),0_Not pollen
4,Dataset/pollen_dataset_kfu_2025\historical_cla...,01 jul (303)_0.png,01 jul (303),0_Not pollen
...,...,...,...,...
5725,Dataset/pollen_dataset_kfu_2025\historical_cla...,september 2 (27)_0.png,september 2 (27),Urtica
5726,Dataset/pollen_dataset_kfu_2025\historical_cla...,september 2 (28)_0.png,september 2 (28),Urtica
5727,Dataset/pollen_dataset_kfu_2025\historical_cla...,september 2 (29)_0.png,september 2 (29),Urtica
5728,Dataset/pollen_dataset_kfu_2025\historical_cla...,september 2 (41)_0.png,september 2 (41),Urtica


In [15]:
hist_predictions_df = process_set(hist_images_ds)
unpack_embeddings_and_save(hist_predictions_df, hist_cls_pred_path)


0it [00:00, ?it/s]